# Segment Heat Map

Sweeps the prior-month and current-month workbooks from the tab after
**Secured Card<=600** through **CSOX $75640+**, adds the tabs named in
`EXTRA_SEGMENTS` that sit before that range, pulls the twelve metrics, and writes a heat
map with **current / prior / variance** for each one (the variance column gets a
green-yellow-red colour scale).

The two months move on their own: sign-offs run a month ahead, so current is the month
*after* today - run it in August 2026 and the columns read **Sep-2026** and **Aug-2026**.
The four file names follow the house convention, `09'26 P&L Forecast - All.xlsm` and
`09'26 P&L Forecast - Rollup.xlsx`, built from those two months; `FOLDER_PATTERN` puts each
month in its own folder, and the four names can be written out literally instead. Pin
`CURRENT_MONTH` to override the months.

`EXTRA_SEGMENTS` is read tab by tab, exactly as named, and written above the swept rows.
Nothing else from before `Secured Card<=600` comes in, and `NameManager` and `Sheet1`,
which follow the last segment tab, are skipped.

Roll-up tabs are named in `ROLLUP_SHEETS`. Two families can share one roll-up:

| segment tabs | roll-up tab |
| --- | --- |
| `CSOX $39` | `Credit Sesame $39_ALL` |
| `AMEX DCO $39` | `AMEX DCO $39_ALL` |
| `BrightMoney $39` | `BrightMoney $39_ALL` |
| `Organic $39` | `ORGANIC $39_ALL` |
| `MoneyLion DCO $39` | `MoneyLion DCO $39_ALL` |
| `Creditcards.com $75` | `CREDITCARDS.COM_ALL` |
| `MoneyLion DCO $75` | `MoneyLion DCO $75_ALL` |
| `Google $75` | `Google PQ Paid Search_ALL` |
| `CSOX $75` | `Credit Sesame $75_ALL` |
| `Credit Karma $95` | `Credit Karma $95_ALL` |
| `CK Wander $95` | `Credit Karma WANDER_ALL` |
| `EXPX Wander $95` | `EXPERIAN WANDER_ALL` |
| `Organic $95` + `Organic Wander $95` | `ORGANIC X5 and Wander_ALL` |
| `CSOX $95` + `CSOX Wander $95` | `Credit Sesame X5 and Wander_ALL` |

`Credit Karma $0_ALL`, `Experian $0_ALL`, `AMEX DCO $0_ALL`, `ORGANIC $0_ALL` and
`Organic Omni $0_ALL` have no segments behind them, so they are written one after the
other at the bottom (`ROLLUP_ONLY`).

Underneath the table sits the campaign-total block, from `CAMPAIGN_TOTALS`:
`Vantage <=600`, `Vantage 601-639`, `Vantage 640+` and `Omni` read the `<=600`, `601-639`,
`640+` and `Omni_ALL` tabs, and a last `Campaign Total` row reads `Digital_ALL`. Segments,
roll-ups and campaign totals are shaded on three separate colour scales.

Rows with no volume in either month are dropped.

Run the cells in order. The only cell you edit is the first one.

In [ ]:
# ── 1. Settings ───────────────────────────────────────────────────────────────
import datetime, math, os
import numpy as np
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

PROJECT = "/Users/evernjoshua/Madam Claude Automation"
os.chdir(PROJECT)

# ── which two months ─────────────────────────────────────────────────────────
# Sign-offs run a month ahead, so CURRENT is the month AFTER today: run this in
# August 2026 and current is Sep-2026, prior Aug-2026. Leave CURRENT_MONTH blank
# to keep that automatic, or pin it - "Sep 2026", "September 2026" and "2026-09"
# all parse.
CURRENT_MONTH = ""
SIGNOFF_LEAD  = 1        # months ahead of today; 0 = today's own calendar month

def _shift_month(d, n):
    m = d.month - 1 + n
    return d.replace(year=d.year + m // 12, month=m % 12 + 1, day=1)

def _parse_month(text):
    for fmt in ("%b %Y", "%B %Y", "%b-%Y", "%B-%Y", "%Y-%m", "%m/%Y", "%b %y"):
        try:
            return datetime.datetime.strptime(str(text).strip(), fmt).date().replace(day=1)
        except ValueError:
            pass
    raise ValueError(f"CURRENT_MONTH={text!r} not understood - try 'Sep 2026'")

CURR_MONTH  = (_parse_month(CURRENT_MONTH) if CURRENT_MONTH
               else _shift_month(datetime.date.today(), SIGNOFF_LEAD))
PRIOR_MONTH = _shift_month(CURR_MONTH, -1)

CURR_LABEL  = CURR_MONTH.strftime("%b-%Y")     # "Sep-2026" - the column heading
PRIOR_LABEL = PRIOR_MONTH.strftime("%b-%Y")    # "Aug-2026"

# File names are built from those two months, in the house naming convention:
#   09'26 P&L Forecast - All.xlsm  /  09'26 P&L Forecast - Rollup.xlsx
# Placeholders: {mm} 09  {m} 9  {yy} 26  {year} 2026  {mon} Sep  {month} September
WORKBOOK_PATTERN = "{mm}'{yy} P&L Forecast - All.xlsm"
ROLLUP_PATTERN   = "{mm}'{yy} P&L Forecast - Rollup.xlsx"

# Each month sits in its own folder, so give the folder the same treatment. It is
# relative to PROJECT unless you write an absolute path; "" means PROJECT itself.
#   e.g. "{mm}'{yy}"  or  "Forecasts/{month} {year}"
FOLDER_PATTERN = ""

def _book(pattern, d):
    keys = dict(mm=f"{d.month:02d}", m=str(d.month), yy=d.strftime("%y"),
                year=d.year, mon=d.strftime("%b"), month=d.strftime("%B"))
    name = pattern.format(**keys)
    folder = FOLDER_PATTERN.format(**keys) if FOLDER_PATTERN else ""
    return os.path.join(folder, name) if folder else name

CURRENT_WORKBOOK = _book(WORKBOOK_PATTERN, CURR_MONTH)     # <-- this month
PRIOR_WORKBOOK   = _book(WORKBOOK_PATTERN, PRIOR_MONTH)    # <-- last month
CURRENT_ROLLUP   = _book(ROLLUP_PATTERN, CURR_MONTH)
PRIOR_ROLLUP     = _book(ROLLUP_PATTERN, PRIOR_MONTH)

# Or name the four files outright - uncomment and edit, and the patterns above
# stop mattering. Paths can be absolute, or relative to PROJECT.
# PRIOR_WORKBOOK   = "08'26 P&L Forecast - All.xlsm"
# CURRENT_WORKBOOK = "09'26 P&L Forecast - All.xlsm"
# PRIOR_ROLLUP     = "08'26 P&L Forecast - Rollup.xlsx"
# CURRENT_ROLLUP   = "09'26 P&L Forecast - Rollup.xlsx"

OUTPUT_FILE = "segment_heatmap.xlsx"

# ── which tabs ───────────────────────────────────────────────────────────────
# The sweep runs from the tab AFTER "Secured Card <=600" through
# "Organic Omni $0_ALL", and everything in between comes along. Leave any of the
# three blank to run to that end of the workbook instead ("" for all three is the
# whole file).
START_AFTER_SHEET = "Secured Card<=600"     # range begins at the tab AFTER this one
START_SHEET       = ""                      # or name the first tab here, and blank the line above
END_SHEET         = "CSOX $75640+"           # the last segment tab; NameManager/Sheet1 follow it

# Tabs from OUTSIDE that range, named one by one and read exactly as they are.
# These sit before "Secured Card <=600", so the sweep never reaches them - but
# nothing else before Secured Card is wanted, which is why they are listed rather
# than the range being widened. They are written above the swept rows, in this
# order, and get their roll-up rows like any other family.
EXTRA_SEGMENTS = [
    "MoneyLion DCO $75 <=600",
    "Creditcards.com $75 640+", "Creditcards.com $75 601-6", "Creditcards.com $75 <=600",
    "CSOX $39 640+", "CSOX $39 601-6", "CSOX $39 <=600",
    "AMEX DCO $39 640+", "AMEX DCO $39 601-6", "AMEX DCO $39 <=600",
    "Organic $39 640+", "Organic $39 601-6", "Organic $39 <=600",
    "MoneyLion DCO $39 640+",
    "CSOX Wander $95 640+", "CSOX Wander $95 601-6", "CSOX Wander $95 <=600",
    "CSOX $95 640+", "CSOX $95 601-6", "CSOX $95 <=600",
    "Organic $95 640+", "Organic $95 601-6", "Organic $95 <=600",
    # No BrightMoney tab exists in the workbook - uncomment when one turns up.
    # "BrightMoney $39 <=600",
]

# Tabs inside the range that are not segments - cover sheets, assumptions, notes.
# Rows with no volume in either month drop out on their own, so this is only for
# a tab that carries numbers you do not want in the heat map.
SKIP_SHEETS = ["NameManager", "Sheet1"]

# A row needs at least MIN_VOLUME in one of the two months to earn a place. Below
# that in BOTH months it drops out of the heat map; SHOW_EXCLUDED then lists it
# under everything else, with no figures, and the Comments column saying why.
HIDE_EMPTY_ROWS = True
MIN_VOLUME      = 1
SHOW_EXCLUDED   = True

# Row order: families are grouped by annual fee, in this order, and inside a fee
# they keep the order they come off the tabs. The $0 slot is where the roll-up-only
# rows land. Any fee not listed here follows the listed ones; campaign totals are
# always last, with the excluded rows after them.
FEE_ORDER = ["$75", "$39", "$0", "$95"]

ROLLUP_SUFFIX = "_ALL"

# Which roll-up tab each family rolls into. The key is the segment name minus its
# band; the value is the tab as the roll-up workbook spells it. Two families can
# share one roll-up - "Credit Sesame X5 and Wander_ALL" covers both CSOX $95 and
# CSOX Wander $95 - and then a single roll-up row is written under the last of
# them. The map is read in reverse too, so an _ALL tab sitting in the MAIN
# workbook is matched back to its family instead of drifting off on its own.
# A family that is not listed keeps the plain rule: family + ROLLUP_SUFFIX.
ROLLUP_SHEETS = {
    # $39
    "CSOX $39":            "Credit Sesame $39_ALL",
    "AMEX DCO $39":        "AMEX DCO $39_ALL",
    "BrightMoney $39":     "BrightMoney $39_ALL",
    "Organic $39":         "ORGANIC $39_ALL",
    "MoneyLion DCO $39":   "MoneyLion DCO $39_ALL",
    # $75
    "Creditcards.com $75": "CREDITCARDS.COM_ALL",
    "MoneyLion DCO $75":   "MoneyLion DCO $75_ALL",
    "Google $75":          "Google PQ Paid Search_ALL",
    "CSOX $75":            "Credit Sesame $75_ALL",
    # $95
    "Credit Karma $95":    "Credit Karma $95_ALL",
    "CK Wander $95":       "Credit Karma WANDER_ALL",
    "EXPX Wander $95":     "EXPERIAN WANDER_ALL",
    "Organic $95":         "ORGANIC X5 and Wander_ALL",
    # "Wander $95" sits just above Organic $95 in the workbook and is presumably the
    # Wander half of that roll-up, but it was not asked for - add its three tabs to
    # EXTRA_SEGMENTS and this line groups them into the same row.
    "Wander $95":          "ORGANIC X5 and Wander_ALL",
    "CSOX $95":            "Credit Sesame X5 and Wander_ALL",
    "CSOX Wander $95":     "Credit Sesame X5 and Wander_ALL",
}

# Roll-ups with no segment tabs behind them. They are written one after the other
# at the BOTTOM of the heat map, in this order, and are looked for in the roll-up
# workbook first and the main workbook second.
ROLLUP_ONLY = [
    "Credit Karma $0_ALL",
    "Experian $0_ALL",
    "AMEX DCO $0_ALL",
    "ORGANIC $0_ALL",
    "Organic Omni $0_ALL",
]

# The campaign-total block, written underneath the heat map on its own scale.
# Left is the row label, right is the tab it is read from - the roll-up workbook
# first, the main workbook second, same as every other roll-up.
CAMPAIGN_TOTALS = [
    ("Vantage <=600",   "<=600"),
    ("Vantage 601-639", "601-639"),
    ("Vantage 640+",    "640+"),
    ("Omni",            "Omni_ALL"),
    ("Campaign Total",  "Digital_ALL"),
]

# The top summary boxes read straight from this tab in the roll-up workbooks,
# using the same cells as everything else. Blank it to fall back to a
# volume-weighted average across the segments instead.
SUMMARY_SHEET = "Digital_ALL"

# Vantage bands, LOWEST first. Two jobs: a sheet name minus its band is the
# family, and the roll-up row is placed under the highest-ranked band a family
# actually has - 640+ if it exists, else 601-6, and so on down. Add any band
# spelling your workbook uses. The space before the band is optional, so both
# "Credit Karma $95 640+" and "Credit Karma $95640+" split the same way.
BANDS = ["<=600", "601-6", "601-639", "630-6", "640+"]

# Where the roll-up row lands inside its group:
#   "bottom" - always after the group's last tab in tab order, whatever its band.
#   "band"   - under the highest band the group has (640+, else 601-6, ...).
# The two agree when a family's tabs run in ascending band order, but plenty of
# them run the other way (CSOX $39 goes 640+, 601-6, <=600), and "band" would
# then drop the roll-up row into the MIDDLE of its own family. "bottom" always
# closes the block, so that is the default.
ROLLUP_AT = "bottom"

# Unit losses, $ losses and ROA are all ratios - none of them carry a $ sign.
# good="up"   -> a positive variance is good  (green high / red low)
# good="down" -> a negative variance is good  (green low  / red high)
#
# NOTE: ROA Yr1 is read from H22 exactly as specified, while Yr2/Yr3 come from
# I23/J23. If ROA actually sits on one row, change "H22" to "H23" here.
METRICS = [
    dict(label="Volume",        cell="B6",  good="up",   fmt="#,##0",          py="{:,.0f}"),
    dict(label="CPA",           cell="B7",  good="down", fmt='"$"#,##0.00',    py="${:,.2f}"),
    dict(label="Avg Credit Line", cell="B9", good="up",   fmt='"$"#,##0',       py="${:,.0f}"),
    dict(label="Yr1 Unit Loss", cell="H11", good="down", fmt="0.00%",          py="{:.2%}"),
    dict(label="Yr2 Unit Loss", cell="I11", good="down", fmt="0.00%",          py="{:.2%}"),
    dict(label="Yr3 Unit Loss", cell="J11", good="down", fmt="0.00%",          py="{:.2%}"),
    dict(label="Yr1 $ Loss",    cell="H12", good="down", fmt="0.00%",          py="{:.2%}"),
    dict(label="Yr2 $ Loss",    cell="I12", good="down", fmt="0.00%",          py="{:.2%}"),
    dict(label="Yr3 $ Loss",    cell="J12", good="down", fmt="0.00%",          py="{:.2%}"),
    dict(label="Yr1 ROA",       cell="H22", good="up",   fmt="0.00%",          py="{:.2%}"),
    dict(label="Yr2 ROA",       cell="I23", good="up",   fmt="0.00%",          py="{:.2%}"),
    dict(label="Yr3 ROA",       cell="J23", good="up",   fmt="0.00%",          py="{:.2%}"),
]

# Current month first, then prior, then the variance. These three keys are
# internal; the headings anyone actually reads are the month labels below.
SUBCOLS = ("Curr", "Prior", "Var")
SUBCOL_LABELS = {"Curr": CURR_LABEL, "Prior": PRIOR_LABEL, "Var": "Variance"}

# Excel's own 3-colour-scale palette, reused for the notebook preview
GREEN, YELLOW, RED = "63BE7B", "FFEB84", "F8696B"

print(f"settings loaded - {CURR_LABEL} against {PRIOR_LABEL}")
print(f"  current  {CURRENT_WORKBOOK!r}  +  {CURRENT_ROLLUP!r}")
print(f"  prior    {PRIOR_WORKBOOK!r}  +  {PRIOR_ROLLUP!r}")
print(f"  {len(METRICS)} metrics, {len(ROLLUP_SHEETS)} mapped families, "
      f"{len(EXTRA_SEGMENTS)} named tabs from outside the range,")
print(f"  {len(ROLLUP_ONLY)} roll-up-only rows, {len(CAMPAIGN_TOTALS)} campaign totals")

## 2. Check the files and the tab names

The sweep runs from the tab after `START_AFTER_SHEET` to `END_SHEET`; the tabs before it
that are still wanted are named in `EXTRA_SEGMENTS`. Copy spellings out of this list for
either of those, or for `SKIP_SHEETS`. Cell 4 prints which tab the range actually starts
on and flags any `EXTRA_SEGMENTS` name it could not find.

In [ ]:
# ── 2. What is actually in the workbooks ──────────────────────────────────────
def list_sheets(path):
    wb = load_workbook(path, read_only=True)
    try:
        return list(wb.sheetnames)
    finally:
        wb.close()

for label, path in [("PRIOR", PRIOR_WORKBOOK), ("CURRENT", CURRENT_WORKBOOK)]:
    ok = os.path.exists(path)
    print(f"{label:8} {'FOUND    ' if ok else 'NOT FOUND'}  {path}")
    if ok:
        for i, name in enumerate(list_sheets(path)):
            print(f"    {i:3}  {name!r}")
    print()

In [ ]:
# ── 3. Readers ────────────────────────────────────────────────────────────────
import re

NAN = float("nan")

def _norm(s):
    # tolerant tab matching: case and repeated whitespace do not matter
    return " ".join(str(s).split()).strip().lower()

def _squash(s):
    # ignore case and every space, so "Credit Karma $75 _ALL" still matches
    return "".join(str(s).split()).lower()

def sheet_span(names, start, end, after=None):
    # any of the three blank means "run to the end of the workbook that side".
    # Matching ignores case AND spacing, so "Secured Card <=600" finds the tab
    # that is actually spelled "Secured Card<=600".
    index = {}
    for i, n in enumerate(names):
        index.setdefault(_squash(n), i)
    if after:
        a = index.get(_squash(after))
        if a is None:
            raise KeyError("START_AFTER_SHEET not found. Tabs present:\n  "
                           + "\n  ".join(names))
        if a + 1 >= len(names):
            raise KeyError(f"START_AFTER_SHEET {after!r} is the last tab - nothing follows it")
        i = a + 1
        print(f"    start after {names[a]!r} -> first tab is {names[i]!r}")
    else:
        i = 0 if not start else index.get(_squash(start))
    j = len(names) - 1 if not end else index.get(_squash(end))
    missing = [lab for lab, k in (("START_SHEET", i), ("END_SHEET", j)) if k is None]
    if missing:
        raise KeyError(
            ", ".join(missing) + " not found. Tabs present:\n  " + "\n  ".join(names)
        )
    if i > j:
        i, j = j, i
    return names[i:j + 1]

def _num(v):
    # cached Excel values come back as numbers, strings, None or errors
    if v is None or isinstance(v, bool):
        return NAN
    if isinstance(v, (int, float)):
        return float(v)
    t = str(v).strip()
    if not t or t.startswith("#"):          # #DIV/0!, #REF! ...
        return NAN
    neg = t.startswith("(") and t.endswith(")")
    t = t.strip("()").replace(",", "").replace("$", "").replace(" ", "")
    pct = t.endswith("%")
    t = t.rstrip("%")
    try:
        x = float(t)
    except ValueError:
        return NAN
    if pct:
        x /= 100.0
    return -x if neg else x

def _metrics(ws):
    return {m["label"]: _num(ws[m["cell"]].value) for m in METRICS}

def read_book(path, start=None, end=None):
    # one row per sheet, one column per metric
    start = START_SHEET if start is None else start
    end   = END_SHEET   if end   is None else end
    wb = load_workbook(path, data_only=True)
    try:
        tabs = sheet_span(wb.sheetnames, start, end, after=START_AFTER_SHEET or None)
        skip = {_squash(s) for s in SKIP_SHEETS}
        tabs = [t for t in tabs if _squash(t) not in skip]
        rows = {name: _metrics(wb[name]) for name in tabs}
    finally:
        wb.close()
    df = pd.DataFrame.from_dict(rows, orient="index")[[m["label"] for m in METRICS]]
    if df.notna().to_numpy().sum() == 0:
        print(f"WARNING: every cell read from {path} was empty. openpyxl reads the "
              f"values Excel cached on last save - open the file in Excel and save it.")
    print(f"{os.path.basename(path)}: {len(df)} sheets ({tabs[0]!r} ... {tabs[-1]!r})")
    return df

def read_named(path, tabs):
    # the tabs listed in EXTRA_SEGMENTS, read exactly as named and kept in the
    # order given - the same shape read_book returns, so the two concatenate
    labels = [m["label"] for m in METRICS]
    if not tabs:
        return pd.DataFrame(columns=labels)
    found = read_tabs(path, tabs)
    missing = [t for t in tabs if t not in found]
    if missing:
        print(f"{os.path.basename(path)}: EXTRA_SEGMENTS not found: {missing}")
    rows = {found[t][0]: found[t][1] for t in tabs if t in found}
    if not rows:
        return pd.DataFrame(columns=labels)
    print(f"{os.path.basename(path)}: {len(rows)} named tabs from outside the range")
    return pd.DataFrame.from_dict(rows, orient="index")[labels]

# ---- roll-ups ---------------------------------------------------------------
# "Credit Karma $75 <=600", "... 630-6" and "... 640+" are one family: same name,
# same annual fee, different vantage band. The family key is the sheet name up to
# and including the fee, and the roll-up tab is whatever ROLLUP_SHEETS says it is.
_FEE = re.compile(r"^(.*?\$\s*\d+)")

# the ROLLUP_SHEETS map, squashed, and its reverse. Two families can share one
# roll-up tab, so the reverse map keeps whichever came last - either answer maps
# forward to the same tab again, which is all rollup_key needs.
ROLLUP_BY_FAMILY = {_squash(k): v for k, v in ROLLUP_SHEETS.items()}
FAMILY_BY_ROLLUP = {_squash(v): k for k, v in ROLLUP_SHEETS.items()}
ROLLUP_ONLY_SQUASHED = {_squash(t) for t in ROLLUP_ONLY}
CAMPAIGN_SQUASHED = {_squash(t) for _, t in CAMPAIGN_TOTALS}
# tabs that are written as their own block, never as a segment row
BLOCK_SQUASHED = ROLLUP_ONLY_SQUASHED | CAMPAIGN_SQUASHED

def split_band(sheet):
    # "EXPX Wander $95 640+" -> ("EXPX Wander $95", "640+"). Keyed off the band and
    # not the fee, because plenty of products have no $ in the name at all.
    # The space in front of the band is optional: "EXPX Wander $95640+" splits the
    # same way. Longest band first, so "<=600" wins over a bare "600" spelling.
    s = " ".join(str(sheet).split())
    low = s.lower()
    for b in sorted(BANDS, key=len, reverse=True):
        if low.endswith(b.lower()):
            return s[:len(s) - len(b)].strip(), b
    return None, None

def group_key(sheet):
    named = FAMILY_BY_ROLLUP.get(_squash(sheet))   # a roll-up tab we have a name for
    if named:
        return named
    fam, _ = split_band(sheet)
    if fam:
        return fam
    m = _FEE.match(str(sheet))              # no band: fall back to the fee
    if m:
        return m.group(1).strip()
    return " ".join(str(sheet).split())     # and failing that, the whole name

def band_rank(sheet):
    # where this sheet's band sits in BANDS; -1 for a band we do not know
    _, b = split_band(sheet)
    return BANDS.index(b) if b in BANDS else -1

def rollup_name(family):
    # the tab this family rolls up into: the name from ROLLUP_SHEETS if it has
    # one, otherwise the old family + "_ALL"
    return ROLLUP_BY_FAMILY.get(_squash(family), f"{family}{ROLLUP_SUFFIX}")

def rollup_key(sheet):
    # the roll-up tab a segment belongs under. This, not the family, is what
    # groups rows: CSOX $95 and CSOX Wander $95 share one roll-up tab and so
    # share one roll-up row.
    return rollup_name(group_key(sheet))

def is_all_sheet(name):
    # an _ALL tab sitting in the main range IS the roll-up row - never pull a
    # second one in from the roll-up file for the same family. A named roll-up
    # such as "Credit Karma WANDER_ALL" counts here too, even though its family
    # is spelled differently, because group_key maps it back to "CK Wander $95".
    sq = _squash(name)
    return (sq in FAMILY_BY_ROLLUP
            or sq in BLOCK_SQUASHED
            or sq.endswith(_squash(ROLLUP_SUFFIX)))

def read_tabs(path, tabs):
    # {tab as asked for: (tab as the file spells it, {metric: value})}
    if not tabs or not path or not os.path.exists(path):
        return {}
    wb = load_workbook(path, data_only=True)
    try:
        index = {}
        for name in wb.sheetnames:
            index.setdefault(_squash(name), name)
        found = {}
        for t in tabs:
            hit = index.get(_squash(t))
            if hit is not None:
                found[t] = (hit, _metrics(wb[hit]))
    finally:
        wb.close()
    return found

def read_rollups(rollup_path, main_path, tabs):
    # the roll-up workbook first, then the main workbook for anything it lacks -
    # some months the _ALL tabs live in one file, some months the other
    if not tabs:
        return {}
    if rollup_path and not os.path.exists(rollup_path):
        print(f"roll-up file not found, trying the main workbook instead: {rollup_path!r}")
    found = read_tabs(rollup_path, tabs)
    rest = [t for t in tabs if t not in found]
    if rest:
        also = read_tabs(main_path, rest)
        if also:
            print(f"    {len(also)} roll-up tab(s) taken from {os.path.basename(main_path)}")
        found.update(also)
    missing = [t for t in tabs if t not in found]
    label = os.path.basename(str(rollup_path or main_path))
    print(f"{label}: matched {len(found)} of {len(tabs)} roll-up tabs"
          + (f" - NOT FOUND: {missing}" if missing else ""))
    return found

def read_summary(path, sheet):
    # the portfolio-level row for the top boxes, read from its own tab
    if not sheet or not path or not os.path.exists(path):
        return None
    wb = load_workbook(path, data_only=True)
    try:
        index = {}
        for n in wb.sheetnames:
            index.setdefault(_squash(n), n)
        hit = index.get(_squash(sheet))
        if hit is None:
            print(f"{os.path.basename(path)}: summary sheet {sheet!r} not found "
                  f"- the boxes will fall back to a volume-weighted average")
            return None
        ws = wb[hit]
        print(f"{os.path.basename(path)}: summary boxes from {hit!r}")
        return _metrics(ws)
    finally:
        wb.close()

print("readers defined - roll-up tabs:")
for _tab in dict.fromkeys(ROLLUP_SHEETS.values()):
    _fams = [k for k, v in ROLLUP_SHEETS.items() if v == _tab]
    print(f"    {_tab:<34} <- {', '.join(_fams)}")
for _tab in ROLLUP_ONLY:
    print(f"    {_tab:<34} <- (roll-up only)")
for _label, _tab in CAMPAIGN_TOTALS:
    print(f"    {_tab:<34} <- (campaign total: {_label})")

In [ ]:
# ── 4. Build the heat map table ───────────────────────────────────────────────
def _read(path):
    # the named tabs first, then the swept range, with anything that turns up in
    # both kept only once
    df = pd.concat([read_named(path, EXTRA_SEGMENTS), read_book(path)])
    return df[~df.index.duplicated(keep="first")]

prior = _read(PRIOR_WORKBOOK)
curr  = _read(CURRENT_WORKBOOK)

# current month drives row order; any prior-only segment is kept at the bottom
segments = list(curr.index) + [s for s in prior.index if s not in set(curr.index)]

# the roll-up-only tabs are written as their own block at the very bottom, so
# pull them out of the tab order even when they sit inside the main range
segments = [s for s in segments if _squash(s) not in BLOCK_SQUASHED]
prior = prior.reindex(segments)
curr  = curr.reindex(segments)

def _empty(x):
    # below the floor, or not a number at all
    return (x is None or (isinstance(x, float) and math.isnan(x))
            or float(x) < MIN_VOLUME)

NO_VOLUME_NOTE = f"Zero volume in both {CURR_LABEL} and {PRIOR_LABEL} - excluded"

# A tab short of MIN_VOLUME in both months says nothing, so it is dropped here
# rather than after the table is built - that keeps cover sheets and dormant
# products out of the roll-up lookup as well, instead of them asking for a roll-up
# tab that was never going to exist.
hidden = []
if HIDE_EMPTY_ROWS:
    hidden = [s for s in segments
              if _empty(curr.at[s, "Volume"]) and _empty(prior.at[s, "Volume"])]
    segments = [s for s in segments if s not in set(hidden)]
    prior = prior.reindex(segments)
    curr  = curr.reindex(segments)

# _ALL tabs already inside the main range: those rows ARE the roll-up
native = {_squash(s) for s in segments if is_all_sheet(s)}

# One roll-up row per roll-up TAB, not per family - CSOX $95 and CSOX Wander $95
# name the same tab and so share one row. It sits under the highest band any of
# its families has (640+ if present, else 601-6, and so on down BANDS); ties, and
# families whose bands are not in BANDS, fall back to the last one in tab order.
want, anchor = [], {}
for pos, name in enumerate(segments):
    if is_all_sheet(name):
        continue
    tab = rollup_key(name)
    rank = (band_rank(name), pos) if ROLLUP_AT == "band" else (0, pos)
    if tab not in anchor or rank > anchor[tab][0]:
        anchor[tab] = (rank, name)
    if tab not in want and _squash(tab) not in native:
        want.append(tab)

roll_prior = read_rollups(PRIOR_ROLLUP,   PRIOR_WORKBOOK,   want)
roll_curr  = read_rollups(CURRENT_ROLLUP, CURRENT_WORKBOOK, want)

extra_prior = read_rollups(PRIOR_ROLLUP,   PRIOR_WORKBOOK,   ROLLUP_ONLY)
extra_curr  = read_rollups(CURRENT_ROLLUP, CURRENT_WORKBOOK, ROLLUP_ONLY)
roll_prior.update(extra_prior)
roll_curr.update(extra_curr)

total_tabs  = [tab for _, tab in CAMPAIGN_TOTALS]
total_prior = read_rollups(PRIOR_ROLLUP,   PRIOR_WORKBOOK,   total_tabs)
total_curr  = read_rollups(CURRENT_ROLLUP, CURRENT_WORKBOOK, total_tabs)
roll_prior.update(total_prior)
roll_curr.update(total_curr)

SUMMARY_PRIOR = read_summary(PRIOR_ROLLUP   or PRIOR_WORKBOOK,   SUMMARY_SHEET)
SUMMARY_CURR  = read_summary(CURRENT_ROLLUP or CURRENT_WORKBOOK, SUMMARY_SHEET)

rows = []
for name in segments:
    if is_all_sheet(name):                      # an _ALL tab in the main range
        rows.append({"label": name, "roll": True, "native": True, "src": name,
                     "kind": "rollup", "block": rollup_key(name)})
        continue
    rows.append({"label": name, "roll": False, "native": False, "src": name,
                 "kind": "segment", "block": rollup_key(name)})
    tab = rollup_key(name)
    if anchor.get(tab, (None, None))[1] != name:    # not this group's anchor row
        continue
    if _squash(tab) in native:                      # its _ALL tab is already in range
        continue
    hit = roll_curr.get(tab) or roll_prior.get(tab)
    if hit:
        rows.append({"label": hit[0], "roll": True, "native": False, "src": tab,
                     "kind": "rollup", "block": tab})

# the roll-up-only rows, one after the other - the $0 group
for tab in ROLLUP_ONLY:
    hit = extra_curr.get(tab) or extra_prior.get(tab)
    if hit:
        rows.append({"label": hit[0], "roll": True, "native": False, "src": tab,
                     "kind": "rollup", "block": tab})

# and the campaign totals, which always sort last
for label, tab in CAMPAIGN_TOTALS:
    if tab in total_curr or tab in total_prior:
        rows.append({"label": label, "roll": True, "native": False, "src": tab,
                     "kind": "total", "block": "\x00totals"})

# ---- order the blocks by annual fee -----------------------------------------
# A block is one family (or the pair that shares a roll-up) plus its roll-up row.
# Blocks move as a unit; inside a fee they keep the order they came off the tabs.
_FEE_ANY = re.compile(r"\$\s*\d+")

def _fee_of(text):
    # take the band off first: "Experian $39601-639" is $39 in the 601-639 band,
    # and reading the digits straight off the name would call it $39601
    family, _ = split_band(text)
    m = _FEE_ANY.search(family or str(text))
    return m.group(0).replace(" ", "") if m else ""

def _fee_rank(fee):
    return FEE_ORDER.index(fee) if fee in FEE_ORDER else len(FEE_ORDER)

first_seen, block_fee = {}, {}
for i, r in enumerate(rows):
    b = r["block"]
    first_seen.setdefault(b, i)
    if not block_fee.get(b):
        block_fee[b] = _fee_of(r["label"]) or _fee_of(b)

def _block_rank(b):
    if b == "\x00totals":
        return (len(FEE_ORDER) + 1, first_seen[b])
    return (_fee_rank(block_fee.get(b, "")), first_seen[b])

grouped = {}
for r in rows:
    grouped.setdefault(r["block"], []).append(r)
rows = [r for b in sorted(grouped, key=_block_rank) for r in grouped[b]]

print("\nrow order by fee:")
for b in sorted(grouped, key=_block_rank):
    if b == "\x00totals":
        print("  campaign totals")
    else:
        print(f"  {block_fee.get(b) or '(no fee)':<8} {b}")

# the tabs the volume floor dropped, listed under everything else with no figures
if SHOW_EXCLUDED:
    for name in hidden:
        rows.append({"label": name, "roll": True, "native": False, "src": None,
                     "kind": "excluded", "block": "\x00excluded",
                     "note": NO_VOLUME_NOTE})

print("\nroll-up placement:")
for tab, (_, name) in anchor.items():
    if _squash(tab) in native:
        where = "already a tab in the main workbook"
    elif tab in roll_curr or tab in roll_prior:
        where = f"under {name!r}"
    else:
        where = "NO ROLL-UP SHEET FOUND"
    print(f"  {tab:<34} {where}")
for tab in ROLLUP_ONLY:
    got = tab in extra_curr or tab in extra_prior
    print(f"  {tab:<34} {'bottom block' if got else 'NO ROLL-UP SHEET FOUND'}")
for label, tab in CAMPAIGN_TOTALS:
    got = tab in total_curr or tab in total_prior
    print(f"  {tab:<34} {'campaign total ' + repr(label) if got else 'NO SHEET FOUND'}")

ROW_KIND  = [r["kind"] for r in rows]
IS_ROLLUP = [r["roll"] for r in rows]

def _cell(row, book, rolls, label):
    if row["kind"] == "excluded":              # listed, but carries no figures
        return NAN
    if row["roll"] and not row["native"]:      # came from the roll-up workbook
        hit = rolls.get(row["src"])
        return NAN if hit is None else hit[1][label]
    try:                                       # a real tab in the main workbook
        return book.at[row["src"], label]
    except KeyError:
        return NAN

columns = [("Segment", ""), ("Comments", "")]
data    = {("Segment", ""): [r["label"] for r in rows],
           ("Comments", ""): [r.get("note", "") for r in rows]}

for m in METRICS:
    lab = m["label"]
    p = np.array([_cell(r, prior, roll_prior, lab) for r in rows], dtype=float)
    c = np.array([_cell(r, curr,  roll_curr,  lab) for r in rows], dtype=float)
    data[(lab, "Prior")] = p
    data[(lab, "Curr")]  = c
    data[(lab, "Var")]   = c - p               # variance is current minus prior
    columns += [(lab, s) for s in SUBCOLS]     # SUBCOLS sets the column order

heat = pd.DataFrame(data, columns=pd.MultiIndex.from_tuples(columns))

# the same rule again, this time catching roll-up rows that came back empty
if HIDE_EMPTY_ROWS and ("Volume", "Curr") in heat.columns:
    keep = [kind == "excluded" or not (_empty(c) and _empty(p))
            for kind, c, p in zip(ROW_KIND, heat[("Volume", "Curr")],
                                  heat[("Volume", "Prior")])]
    hidden += [lab for lab, k in zip(heat[("Segment", "")], keep) if not k]
    heat = heat.loc[keep].reset_index(drop=True)
    IS_ROLLUP = [r for r, k in zip(IS_ROLLUP, keep) if k]
    ROW_KIND  = [r for r, k in zip(ROW_KIND,  keep) if k]
if hidden:
    print(f"\nexcluded - under {MIN_VOLUME} volume in both {CURR_LABEL} and "
          f"{PRIOR_LABEL} ({len(hidden)}):")
    for lab in hidden:
        print(f"  {lab}")

print(f"\n{len(heat)} rows = {ROW_KIND.count('segment')} segments"
      f" + {ROW_KIND.count('rollup')} roll-ups"
      f" + {ROW_KIND.count('total')} campaign totals"
      f" + {ROW_KIND.count('excluded')} excluded")
heat.head()

## 5. Colour it in the notebook

Same colour logic as the Excel file: green = the variance moved the right way for that metric.

In [ ]:
# ── 5. Coloured preview (pure HTML - no matplotlib, no jinja2) ───────────────
from IPython.display import HTML, display

def _rgb(h):
    return tuple(int(h[k:k + 2], 16) for k in (0, 2, 4))

def _mix(a, b, t):
    ra, rb = _rgb(a), _rgb(b)
    return "#%02X%02X%02X" % tuple(int(round(ra[k] + (rb[k] - ra[k]) * t)) for k in range(3))

def _band(t):
    # t = 0 -> green (good), 0.5 -> yellow, 1 -> red (bad)
    t = min(max(t, 0.0), 1.0)
    return _mix(GREEN, YELLOW, t * 2) if t < 0.5 else _mix(YELLOW, RED, t * 2 - 1)

def _colours(values, good, mask=None):
    # mask picks the rows that SET the scale. Segments are shaded against
    # segments, roll-ups against roll-ups and campaign totals against campaign
    # totals - a roll-up volume is a sum of its family and a campaign total a sum
    # of everything, so one shared scale would flatten every segment.
    use = [x for i, x in enumerate(values)
           if (mask is None or mask[i])
           and not (x is None or (isinstance(x, float) and math.isnan(x)))]
    if not use:
        return ["" for _ in values]
    lo, hi = min(use), max(use)
    out = []
    for i, x in enumerate(values):
        if x is None or (isinstance(x, float) and math.isnan(x)) or (mask is not None and not mask[i]):
            out.append("")
            continue
        t = 0.5 if hi == lo else (float(x) - lo) / (hi - lo)   # 0 = lowest value
        if good == "up":                                       # high value is good
            t = 1.0 - t
        out.append(_band(min(max(t, 0.0), 1.0)))
    return out

def _kind_scales(values, good, kinds):
    # one scale per kind of row, each row taking the colour from its own
    out = [""] * len(values)
    for kind in dict.fromkeys(kinds):
        mask = [k == kind for k in kinds]
        got = _colours(values, good, mask)
        for i, m in enumerate(mask):
            if m:
                out[i] = got[i]
    return out

def _txt(x, py):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    return py.format(x)

def heat_html(df, kinds=None):
    kinds = ["segment"] * len(df) if kinds is None else list(kinds)
    bands = {m["label"]: _kind_scales(list(df[(m["label"], "Var")]), m["good"], kinds)
             for m in METRICS}
    th = ("padding:4px 8px;border:1px solid #BFBFBF;background:#1F3864;color:#fff;"
          "font:600 11px/1.3 -apple-system,Segoe UI,sans-serif;text-align:center")
    th2 = th.replace("#1F3864", "#D9E1F2").replace("color:#fff", "color:#1F3864")
    td = ("padding:3px 8px;border:1px solid #E2E2E2;color:#1F2933;"
          "font:11px/1.3 -apple-system,Segoe UI,sans-serif;text-align:right;white-space:nowrap")

    h = ["<div style='overflow:auto;max-height:640px'><table style='border-collapse:collapse'>"]
    h.append("<tr><th rowspan=2 style='%s'>Segment</th><th rowspan=2 style='%s'>Comments</th>" % (th, th))
    for m in METRICS:
        h.append("<th colspan=3 style='%s'>%s</th>" % (th, m["label"]))
    h.append("</tr><tr>")
    for m in METRICS:
        for sub in SUBCOLS:
            h.append("<th style='%s'>%s</th>" % (th2, SUBCOL_LABELS[sub]))
    h.append("</tr>")

    first_total = kinds.index("total") if "total" in kinds else -1
    first_excl  = kinds.index("excluded") if "excluded" in kinds else -1
    for r in range(len(df)):
        kind = kinds[r]
        roll = kind != "segment"
        edge = ("border-top:3px solid #1F3864" if r in (first_total, first_excl)
                else ("border-top:2px solid #9AA0B4" if roll else ""))
        h.append("<tr>")
        h.append("<td style='%s;text-align:left;font-weight:%s;%s%s'>%s</td>"
                 % (td, "400" if kind == "excluded" else ("700" if roll else "600"),
                    edge, ";color:#6B7280" if kind == "excluded" else "",
                    df[("Segment", "")].iloc[r]))
        note = df[("Comments", "")].iloc[r]
        h.append("<td style='%s;min-width:150px;text-align:left;%s'>%s</td>"
                 % (td, "color:#6B7280;font-style:italic" if note else "",
                    note if note else ""))
        for m in METRICS:
            for sub in SUBCOLS:
                x = df[(m["label"], sub)].iloc[r]
                bg = bands[m["label"]][r] if sub == "Var" else ""
                style = td + (";background:%s" % bg if bg else "") + (";" + edge if edge else "")
                h.append("<td style='%s'>%s</td>" % (style, _txt(x, m["py"])))
        h.append("</tr>")
    h.append("</table></div>")
    return "".join(h)

display(HTML(heat_html(heat, ROW_KIND)))

In [ ]:
# ── 6. Write the Excel heat map ───────────────────────────────────────────────

def _v(x):
    return None if x is None or (isinstance(x, float) and math.isnan(x)) else float(x)

def _runs(nums):
    # [3,4,5,7,8] -> [(3,5),(7,8)], so the colour scale can span a broken set of rows
    out = []
    for n in sorted(nums):
        if out and n == out[-1][1] + 1:
            out[-1][1] = n
        else:
            out.append([n, n])
    return [tuple(r) for r in out]

def write_heatmap(df, path=None, kinds=None):
    path = OUTPUT_FILE if path is None else path
    kinds = ["segment"] * len(df) if kinds is None else list(kinds)
    wb = Workbook()
    ws = wb.active
    ws.title = "Heat Map"

    head_fill = PatternFill("solid", fgColor="1F3864")
    sub_fill  = PatternFill("solid", fgColor="D9E1F2")
    head_font = Font(bold=True, color="FFFFFF", size=11)
    sub_font  = Font(bold=True, color="1F3864", size=10)
    thin      = Side(style="thin", color="BFBFBF")
    border    = Border(left=thin, right=thin, top=thin, bottom=thin)
    centre    = Alignment(horizontal="center", vertical="center", wrap_text=True)

    # header rows 1-2
    for col, title in ((1, "Segment"), (2, "Comments")):
        c = ws.cell(1, col, title)
        c.fill, c.font, c.alignment, c.border = head_fill, head_font, centre, border
        ws.merge_cells(start_row=1, start_column=col, end_row=2, end_column=col)
        ws.cell(2, col).fill, ws.cell(2, col).border = head_fill, border

    col = 3
    var_cols = []
    for m in METRICS:
        c = ws.cell(1, col, m["label"])
        c.fill, c.font, c.alignment = head_fill, head_font, centre
        ws.merge_cells(start_row=1, start_column=col, end_row=1, end_column=col + 2)
        for k in range(3):
            ws.cell(1, col + k).border = border
            s = ws.cell(2, col + k, SUBCOL_LABELS[SUBCOLS[k]])
            s.fill, s.font, s.alignment, s.border = sub_fill, sub_font, centre, border
        var_cols.append((col + SUBCOLS.index("Var"), m))
        col += 3
    last_col = col - 1

    # data
    roll_fill  = PatternFill("solid", fgColor="EDEFF7")
    total_fill = PatternFill("solid", fgColor="DCE3F5")
    top_rule   = Side(style="medium", color="8C93A8")
    block_rule = Side(style="thick", color="1F3864")

    first_total = kinds.index("total") if "total" in kinds else -1
    first_excl  = kinds.index("excluded") if "excluded" in kinds else -1
    grey        = Font(italic=True, size=10, color="6B7280")

    for r in range(len(df)):
        row = 3 + r
        kind = kinds[r]
        roll = kind != "segment"
        top = block_rule if r in (first_total, first_excl) else (top_rule if roll else thin)
        edge = Border(left=thin, right=thin, bottom=thin, top=top)
        fill = (None if kind == "excluded"
                else (total_fill if kind == "total" else (roll_fill if roll else None)))
        seg = ws.cell(row, 1, df[("Segment", "")].iloc[r])
        seg.border = edge
        seg.font = (grey if kind == "excluded"
                    else Font(bold=True, size=10, color="1F3864" if roll else "000000"))
        seg.alignment = Alignment(vertical="center", wrap_text=True)
        if fill:
            seg.fill = fill
        note = df[("Comments", "")].iloc[r]             # blank unless the row carries one
        com = ws.cell(row, 2, note or None)
        com.border, com.alignment = edge, Alignment(vertical="top", wrap_text=True)
        if note:
            com.font = grey
        if fill:
            com.fill = fill
        col = 3
        for m in METRICS:
            for k, sub in enumerate(SUBCOLS):
                c = ws.cell(row, col + k, _v(df[(m["label"], sub)].iloc[r]))
                c.number_format = m["fmt"]
                c.border = edge
                c.alignment = Alignment(horizontal="right")
                if kind == "excluded":
                    c.font = grey
                elif roll:
                    c.font = Font(bold=True, size=10)
            col += 3
    last_row = 2 + len(df)

    # green -> yellow -> red on every variance column, flipped per metric.
    # Segments, roll-ups and campaign totals each get their own scale: a roll-up
    # restates the family above it and a campaign total restates the whole book,
    # so one shared scale would squash the segments they summarise.
    for c, m in var_cols:
        L = get_column_letter(c)
        low, high = (RED, GREEN) if m["good"] == "up" else (GREEN, RED)
        for kind in ("segment", "rollup", "total"):
            group = [3 + r for r in range(len(df)) if kinds[r] == kind]
            if not group:
                continue
            sqref = " ".join(f"{L}{a}:{L}{b}" for a, b in _runs(group))
            ws.conditional_formatting.add(
                sqref,
                ColorScaleRule(start_type="min", start_color=low,
                               mid_type="percentile", mid_value=50, mid_color=YELLOW,
                               end_type="max", end_color=high),
            )

    ws.column_dimensions["A"].width = 26
    ws.column_dimensions["B"].width = 34
    for c in range(3, last_col + 1):
        ws.column_dimensions[get_column_letter(c)].width = 13
    ws.row_dimensions[1].height = 22
    ws.row_dimensions[2].height = 18
    ws.freeze_panes = "C3"
    ws.auto_filter.ref = f"A2:{get_column_letter(last_col)}{last_row}"
    ws.sheet_view.showGridLines = False

    wb.save(path)
    return os.path.abspath(path)

out = write_heatmap(heat, kinds=ROW_KIND)
print("written:", out)

### Handy extras

```python
heat.to_csv("segment_heatmap.csv", index=False)          # flat copy

# only the variances, worst first on Yr1 $ Loss
heat[[("Segment", "")] + [(m["label"], "Var") for m in METRICS]] \
    .sort_values(("Yr1 $ Loss", "Var"), ascending=False)

# a different tab range without touching the settings cell
p = read_book(PRIOR_WORKBOOK, start="MoneyLion $75 601-6", end="Organic $95 <=600")
```

## 7. The shareable page

Writes the heat map as a web page. Two files come out of the same content:

- **`segment_heatmap.html`** - a standalone page. Open it, mail it, print it to PDF.
- **`segment_heatmap_artifact.html`** - the same page minus the `<html>`/`<body>`
  wrapper, which is the form the Artifact publisher wants when you turn it into a
  link.

The page keeps the workbook's green-yellow-red scale, but those two ends are only
2.9 OKLab dE apart under deuteranopia (the floor is 6.0), so a red-green colourblind
reader cannot separate them. Two things make up for it: every variance carries a signed
number, so the direction of the move reads without colour, and a toggle switches to a
blue-grey-red ramp that clears the gate at 12.2. Column headings sort.

In [ ]:
# ── 7. Shareable HTML page ────────────────────────────────────────────────────
# Writes two files from the same content:
#   segment_heatmap.html          - standalone page, open it or mail it round
#   segment_heatmap_artifact.html - same page without the <html>/<body> wrapper,
#                                   which is the form the Artifact publisher wants
import datetime, json
from html import escape

# the top header tier groups metrics by unit, because they are read by unit
FAMILIES = [
    ("Acquisition",     ["Volume", "CPA", "Avg Credit Line"]),
    ("Unit losses",     ["Yr1 Unit Loss", "Yr2 Unit Loss", "Yr3 Unit Loss"]),
    ("Dollar losses",   ["Yr1 $ Loss", "Yr2 $ Loss", "Yr3 $ Loss"]),
    ("Return on assets", ["Yr1 ROA", "Yr2 ROA", "Yr3 ROA"]),
]

# green -> yellow -> red is the workbook's scale and stays the default. Its two
# poles are 2.9 OKLab dE apart under deuteranopia (the floor is 6.0), so every
# variance is written as a signed number, and the page offers a second
# ramp - blue -> grey -> red, 12.2 dE - that clears the gate.
RAMPS = {
    "excel": ["#63BE7B", "#FFEB84", "#F8696B"],
    "safe":  ["#8FBDEE", "#EFEEEA", "#F09B99"],
}

def _groups():
    by_label = {m["label"]: m for m in METRICS}
    out, used = [], set()
    for name, labels in FAMILIES:
        got = [by_label[l] for l in labels if l in by_label]
        if got:
            out.append((name, got))
            used.update(m["label"] for m in got)
    rest = [m for m in METRICS if m["label"] not in used]
    if rest:
        out.append(("Other", rest))
    return out

NAN_ = float("nan")

def _isna(x):
    return x is None or (isinstance(x, float) and math.isnan(x))

def _tvals(values, good, mask=None):
    # 0.0 = the best move in this column, 1.0 = the worst. Same relative scaling
    # Excel's min/50th-percentile/max colour scale uses. `mask` picks the rows
    # that set the scale, so a roll-up - which aggregates the rows above it -
    # never stretches the scale its own segments are read on.
    real = [x for i, x in enumerate(values)
            if (mask is None or mask[i]) and not _isna(x)]
    if not real:
        return [None] * len(values)
    lo, hi = min(real), max(real)
    out = []
    for i, x in enumerate(values):
        if _isna(x) or (mask is not None and not mask[i]):
            out.append(None)
            continue
        t = 0.5 if hi == lo else (float(x) - lo) / (hi - lo)
        out.append(min(max(1.0 - t if good == "up" else t, 0.0), 1.0))
    return out

def _tvals_split(values, good, kinds):
    # one scale per kind of row - segments, roll-ups, campaign totals
    out = [None] * len(values)
    for kind in dict.fromkeys(kinds):
        mask = [k == kind for k in kinds]
        got = _tvals(values, good, mask)
        for i, m in enumerate(mask):
            if m:
                out[i] = got[i]
    return out

def _signed(x, py):
    if _isna(x):
        return "-"
    body = py.format(abs(x))
    return ("+" if x > 0 else "−" if x < 0 else "") + body

def _plain(x, py):
    return "-" if _isna(x) else py.format(x)

def _sum(series):
    real = [x for x in series if not _isna(x)]
    return sum(real) if real else float("nan")

_CSS = """
:root{
  color-scheme: light;
  --bg:#F7F7FA; --panel:#FFFFFF; --panel-2:#FAFAFD;
  --ink:#151824; --ink-2:#5B6175; --ink-3:#888EA1;
  --rule:#E5E6EE; --rule-2:#D2D4E0;
  --accent:#2E3A59; --on-accent:#FFFFFF; --chip:#EDEFF7;
  --cell-ink:#1F2933; --pos:#1B7F4B; --neg:#B3261E; --roll-bg:#F1F2F9; --total-bg:#E4E9F7;
  --s-prior:#2a78d6; --s-curr:#eb6834;
  --flag-bg:#FFF6DE; --flag-ink:#4A3A00; --flag-rule:#E7CB74;
  --shadow:0 1px 2px rgba(20,24,40,.05), 0 10px 28px -16px rgba(20,24,40,.22);
}
@media (prefers-color-scheme: dark){
  :root:not([data-theme="light"]){
    color-scheme: dark;
    --bg:#0F111A; --panel:#171A25; --panel-2:#1C202C;
    --ink:#ECEEF5; --ink-2:#A2A9BC; --ink-3:#767D91;
    --rule:#272B39; --rule-2:#343A4A;
    --accent:#9FB2DE; --on-accent:#12141C; --chip:#232839;
    --cell-ink:#1F2933; --pos:#5FCB8E; --neg:#F08A84; --roll-bg:#20242F; --total-bg:#2A3040;
    --s-prior:#3987e5; --s-curr:#d95926;
    --flag-bg:#2A2412; --flag-ink:#EFDDA4; --flag-rule:#574A20;
    --shadow:0 1px 2px rgba(0,0,0,.4), 0 10px 28px -16px rgba(0,0,0,.7);
  }
}
:root[data-theme="dark"]{
  color-scheme: dark;
  --bg:#0F111A; --panel:#171A25; --panel-2:#1C202C;
  --ink:#ECEEF5; --ink-2:#A2A9BC; --ink-3:#767D91;
  --rule:#272B39; --rule-2:#343A4A;
  --accent:#9FB2DE; --on-accent:#12141C; --chip:#232839;
  --cell-ink:#1F2933; --pos:#5FCB8E; --neg:#F08A84; --roll-bg:#20242F; --total-bg:#2A3040;
  --s-prior:#3987e5; --s-curr:#d95926;
  --flag-bg:#2A2412; --flag-ink:#EFDDA4; --flag-rule:#574A20;
  --shadow:0 1px 2px rgba(0,0,0,.4), 0 10px 28px -16px rgba(0,0,0,.7);
}

*{box-sizing:border-box}
body{
  margin:0; background:var(--bg); color:var(--ink);
  font:400 15px/1.55 "IBM Plex Sans", ui-sans-serif, -apple-system, "Segoe UI", sans-serif;
  -webkit-font-smoothing:antialiased;
}
.wrap{max-width:1460px; margin:0 auto; padding:clamp(20px,4vw,52px) clamp(14px,3vw,36px) 72px;
      display:flex; flex-direction:column; gap:26px}
.sr{position:absolute; width:1px; height:1px; margin:-1px; padding:0; border:0;
    overflow:hidden; clip-path:inset(50%); white-space:nowrap}
:where(a,button,select,summary):focus-visible{outline:2px solid var(--accent); outline-offset:2px; border-radius:3px}

/* masthead */
.mast{display:flex; flex-direction:column; gap:10px;
      border-bottom:1px solid var(--rule-2); padding-bottom:20px}
.eyebrow{font-size:11px; font-weight:600; letter-spacing:.14em; text-transform:uppercase;
         color:var(--accent)}
h1{margin:0; font-family:"Newsreader", ui-serif, Georgia, serif; font-weight:600;
   font-size:clamp(1.95rem,1.2rem+2.1vw,2.85rem); line-height:1.05; letter-spacing:-.015em;
   text-wrap:balance}
.dek{margin:0; color:var(--ink-2); font-size:14.5px; max-width:68ch}
.dek b{color:var(--ink); font-weight:600}

.flag{display:flex; gap:10px; align-items:flex-start; padding:12px 15px; border-radius:8px;
      background:var(--flag-bg); color:var(--flag-ink); border:1px solid var(--flag-rule);
      font-size:13.5px}
.flag b{font-weight:600}

/* stat strip */
.stats{display:grid; grid-template-columns:repeat(auto-fit,minmax(200px,1fr)); gap:12px}
.stat{background:var(--panel); border:1px solid var(--rule); border-radius:10px;
      padding:14px 16px; display:flex; flex-direction:column; gap:5px; box-shadow:var(--shadow)}
.stat .k{font-size:11px; font-weight:600; letter-spacing:.09em; text-transform:uppercase; color:var(--ink-3)}
.stat .v{font-size:23px; font-weight:600; letter-spacing:-.02em; font-variant-numeric:tabular-nums}
.stat .d{font-size:12.5px; color:var(--ink-2); font-variant-numeric:tabular-nums}
.stat .w{font-size:10.5px; color:var(--ink-3); letter-spacing:.02em}
.stat .d.up{color:var(--pos)} .stat .d.down{color:var(--neg)}

/* controls */
.bar{display:flex; flex-wrap:wrap; gap:14px; align-items:center; justify-content:space-between}
.ctl{display:flex; gap:9px; align-items:center; font-size:13px; color:var(--ink-2)}
.seg{display:inline-flex; background:var(--chip); border:1px solid var(--rule); border-radius:8px; padding:2px}
.seg button{appearance:none; border:0; background:transparent; color:var(--ink-2); cursor:pointer;
            font:inherit; font-size:12.5px; font-weight:500; padding:5px 11px; border-radius:6px}
.seg button[aria-pressed="true"]{background:var(--panel); color:var(--ink); box-shadow:var(--shadow)}
select{font:inherit; font-size:12.5px; padding:6px 9px; border-radius:8px; color:var(--ink);
       background:var(--panel); border:1px solid var(--rule)}

/* matrix */
.panel{background:var(--panel); border:1px solid var(--rule); border-radius:12px;
       box-shadow:var(--shadow); overflow:hidden}
.scroll{overflow:auto; max-height:76vh}
table{border-collapse:separate; border-spacing:0; width:100%; font-variant-numeric:tabular-nums}
th,td{white-space:nowrap; border-bottom:1px solid var(--rule); border-right:1px solid var(--rule)}
thead th{position:sticky; background:var(--panel-2); z-index:2; font-weight:600}
tr:last-child td{border-bottom:0}
th.fam{top:0; height:34px; padding:0 12px; text-align:center; font-size:11px;
       letter-spacing:.11em; text-transform:uppercase; color:var(--accent);
       border-bottom:1px solid var(--rule-2)}
th.met{top:34px; height:30px; padding:0 12px; text-align:center; font-size:11.5px;
       color:var(--ink); border-bottom:1px solid var(--rule)}
th.sub{top:64px; height:28px; padding:0 12px; font-size:11px; letter-spacing:.03em;
       font-weight:500; color:var(--ink-2); text-align:right;
       border-bottom:1px solid var(--rule-2)}
th.corner{left:0; z-index:4}
th.fam.corner, th.sub.corner{background:var(--panel-2)}
.stick{position:sticky; left:0; background:var(--panel); z-index:3;
       border-right:1px solid var(--rule-2)}
th.sortable{cursor:pointer; user-select:none}
th.sortable:hover{color:var(--ink)}
th.sortable .arrow{opacity:0; font-size:9px; margin-left:3px}
th.sortable[data-dir] .arrow{opacity:1; color:var(--accent)}
td{padding:0 12px; height:33px; font-size:12.5px}
td.seg{font-weight:600; font-size:13px; text-align:left; min-width:190px; max-width:280px;
       white-space:normal; line-height:1.3; padding:7px 14px}
td.note{min-width:150px; background:var(--panel-2)}
td.num{text-align:right; color:var(--ink-2)}
td.cur{color:var(--ink); font-weight:500}
td.var{position:relative; /* contains the .sr span - without this the page scrolls sideways */
       text-align:right; color:var(--cell-ink); font-weight:600; font-feature-settings:"tnum"}
td.var.blank{background:transparent!important; color:var(--ink-3); font-weight:400}
tr.roll td{background:var(--roll-bg); border-top:2px solid var(--rule-2); font-weight:600}
tr.roll td.stick{background:var(--roll-bg)}
tr.roll td.seg{color:var(--accent); white-space:nowrap; max-width:none}
tr.total td{background:var(--total-bg)}
tr.total td.stick{background:var(--total-bg)}
tr.total.first td{border-top:3px solid var(--accent)}
tr.gone td{color:var(--ink-3); font-style:italic}
tr.gone td.seg{color:var(--ink-3); font-weight:500}
tr.gone.first td{border-top:3px solid var(--accent)}
tr.gone td.note{white-space:nowrap}
.chip{display:inline-block; margin-left:7px; padding:1px 6px; border-radius:99px;
      background:var(--chip); color:var(--ink-2); font-size:9.5px; font-weight:600;
      letter-spacing:.07em; text-transform:uppercase; vertical-align:1px}
tbody tr:hover td:not(.var){background:var(--panel-2)}
tbody tr:hover td.stick{background:var(--panel-2)}
.fam-edge{border-left:1px solid var(--rule-2)!important}

/* small multiples */
.charts{display:flex; flex-direction:column; gap:16px}
.chead{display:flex; flex-wrap:wrap; gap:14px; align-items:flex-end; justify-content:space-between}
.chead h2{margin:0 0 3px; font-family:"Newsreader", ui-serif, Georgia, serif;
          font-size:20px; font-weight:600; letter-spacing:-.01em; color:var(--ink);
          text-transform:none}
.chead p{margin:0; font-size:12.5px; color:var(--ink-2); max-width:62ch}
.chead label{font-size:13px; color:var(--ink-2)}
.key{display:inline-flex; align-items:center; gap:6px; font-size:12px; color:var(--ink-2);
     margin-left:6px}
.sw{width:10px; height:10px; border-radius:3px; display:inline-block; margin-left:8px}
.sw.prior{background:var(--s-prior)} .sw.curr{background:var(--s-curr)}
.panels{display:grid; grid-template-columns:repeat(auto-fill,minmax(280px,1fr)); gap:10px;
        background:var(--panel); border:1px solid var(--rule); border-radius:12px;
        padding:14px; box-shadow:var(--shadow)}
.panels[hidden]{display:none}
.panels svg{width:100%; height:auto; overflow:visible}
.pt{fill:var(--ink); font:600 12px "IBM Plex Sans",sans-serif}
.tick{fill:var(--ink-3); font:10px "IBM Plex Sans",sans-serif; text-anchor:end;
      font-variant-numeric:tabular-nums}
.band{fill:var(--ink-2); font:10.5px "IBM Plex Sans",sans-serif; text-anchor:middle}
.grid{stroke:var(--rule); stroke-width:1}
.zero{stroke:var(--rule-2); stroke-width:1.5}
.bar.prior{fill:var(--s-prior)} .bar.curr{fill:var(--s-curr)}
.bar{transition:opacity .12s}
.panels:hover .bar{opacity:.55}
.panels .bar:hover{opacity:1}
@media (prefers-reduced-motion:reduce){.bar{transition:none}}

/* legend + method */
.foot{display:grid; grid-template-columns:repeat(auto-fit,minmax(280px,1fr)); gap:22px; align-items:start}
.legend{display:flex; flex-direction:column; gap:9px}
.legend h2, .method h2{margin:0; font-size:11px; font-weight:600; letter-spacing:.11em;
                       text-transform:uppercase; color:var(--ink-3)}
.ramp{height:12px; border-radius:99px; border:1px solid var(--rule)}
.ramp-lab{display:flex; justify-content:space-between; font-size:12px; color:var(--ink-2)}
.legend p{margin:0; font-size:12.5px; color:var(--ink-2); line-height:1.5}
.method{font-size:12.5px; color:var(--ink-2)}
.method table{width:auto; font-size:12px}
.method td{border:0; height:22px; padding:0 14px 0 0; white-space:nowrap}
.method td.a{color:var(--ink)}
code{font-family:"IBM Plex Mono", ui-monospace, SFMono-Regular, Menlo, monospace; font-size:11.5px;
     background:var(--chip); padding:1px 5px; border-radius:4px; color:var(--ink)}
.sig{border-top:1px solid var(--rule); padding-top:16px; font-size:12px; color:var(--ink-3);
     display:flex; flex-wrap:wrap; gap:6px 18px}

@media (max-width:640px){
  td.seg{min-width:150px; font-size:12px}
  .scroll{max-height:70vh}
}
@media print{
  .bar,.flag{display:none}
  .scroll{max-height:none; overflow:visible}
  thead th{position:static}
  .stick{position:static}
  body{background:#fff}
  *{-webkit-print-color-adjust:exact; print-color-adjust:exact}
}
@media (prefers-reduced-motion:reduce){*{animation:none!important; transition:none!important}}
"""

_JS = """
(function(){
  var RAMPS = %RAMPS%;
  function rgb(h){h=h.replace('#','');return [0,2,4].map(function(i){return parseInt(h.slice(i,i+2),16)})}
  function mix(a,b,t){var x=rgb(a),y=rgb(b);
    return '#'+[0,1,2].map(function(i){
      var v=Math.round(x[i]+(y[i]-x[i])*t).toString(16);return v.length<2?'0'+v:v}).join('')}
  function band(ramp,t){t=Math.min(Math.max(t,0),1);
    return t<0.5?mix(ramp[0],ramp[1],t*2):mix(ramp[1],ramp[2],t*2-1)}

  var cells=[].slice.call(document.querySelectorAll('td.var[data-t]'));
  function paint(which){
    var ramp=RAMPS[which];
    cells.forEach(function(c){c.style.background=band(ramp,parseFloat(c.dataset.t))});
    document.querySelectorAll('.ramp').forEach(function(el){
      el.style.background='linear-gradient(90deg,'+ramp[0]+','+ramp[1]+','+ramp[2]+')'});
    document.querySelectorAll('.seg [data-ramp]').forEach(function(b){
      b.setAttribute('aria-pressed', b.dataset.ramp===which?'true':'false')});
    try{localStorage.setItem('heatmap-ramp',which)}catch(e){}
  }
  document.querySelectorAll('.seg [data-ramp]').forEach(function(b){
    b.addEventListener('click',function(){paint(b.dataset.ramp)})});
  var saved=null; try{saved=localStorage.getItem('heatmap-ramp')}catch(e){}
  paint(RAMPS[saved]?saved:'excel');

  var tbody=document.querySelector('tbody');
  var original=[].slice.call(tbody.rows);
  document.querySelectorAll('th.sortable').forEach(function(th){
    th.addEventListener('click',function(){
      var i=+th.dataset.col, dir=th.dataset.dir==='desc'?'asc':'desc';
      document.querySelectorAll('th.sortable').forEach(function(o){o.removeAttribute('data-dir')});
      th.dataset.dir=dir;
      var rows=[].slice.call(tbody.rows);
      rows.sort(function(a,b){
        var x=a.cells[i].dataset.v, y=b.cells[i].dataset.v;
        if(x===undefined&&y===undefined) return 0;
        if(x===undefined||x==='') return 1;
        if(y===undefined||y==='') return -1;
        if(isNaN(parseFloat(x))) return String(x).localeCompare(String(y))*(dir==='asc'?1:-1);
        return (parseFloat(x)-parseFloat(y))*(dir==='asc'?1:-1);
      });
      rows.forEach(function(r){tbody.appendChild(r)});
    });
    th.setAttribute('tabindex','0');
    th.addEventListener('keydown',function(e){if(e.key==='Enter'||e.key===' '){e.preventDefault();th.click()}});
  });
  var picker=document.getElementById('metric');
  if(picker) picker.addEventListener('change',function(){
    document.querySelectorAll('.panels').forEach(function(p){
      p.hidden = p.dataset.metric !== picker.value});
  });

  var reset=document.getElementById('reset');
  if(reset) reset.addEventListener('click',function(){
    document.querySelectorAll('th.sortable').forEach(function(o){o.removeAttribute('data-dir')});
    original.forEach(function(r){tbody.appendChild(r)});
  });
})();
"""

def _hx(h):
    h = h.lstrip("#")
    return [int(h[i:i + 2], 16) for i in (0, 2, 4)]

def _blend(a, b, t):
    x, y = _hx(a), _hx(b)
    return "#%02X%02X%02X" % tuple(int(round(x[i] + (y[i] - x[i]) * t)) for i in range(3))

def _fill(t, ramp="excel"):
    a, b, c = RAMPS[ramp]
    t = min(max(t, 0.0), 1.0)
    return _blend(a, b, t * 2) if t < 0.5 else _blend(b, c, t * 2 - 1)

# ---- small multiples: one panel per product, its vantage bands as bars -------
# Prior and current are the first two categorical slots, blue and orange: 24.7
# OKLab dE apart under the worst CVD simulation in light mode, 26.8 in dark, so
# the pair never depends on hue alone being distinguishable.
SERIES = {"prior": ("#2a78d6", "#3987e5"), "curr": ("#eb6834", "#d95926")}

def _band_label(sheet, key):
    # "Credit Karma $75 630-6" minus the family key leaves "630-6"
    tail = str(sheet)[len(key):].strip() if key and str(sheet).startswith(key) else str(sheet)
    return tail or str(sheet)

def _families(df, isroll):
    # {product: [(band label, row index), ...]} for the segment rows only
    out = []
    for i in range(len(df)):
        if isroll[i]:
            continue
        name = str(df[("Segment", "")].iloc[i])
        k = group_key(name) or name
        hit = next((g for g in out if g[0] == k), None)
        if hit is None:
            hit = (k, [])
            out.append(hit)
        hit[1].append((_band_label(name, k), i))
    return out

def _nice_ticks(lo, hi, count=4):
    if hi == lo:
        hi = lo + 1.0
    raw = (hi - lo) / count
    mag = 10.0 ** math.floor(math.log10(abs(raw))) if raw else 1.0
    step = min([m * mag for m in (1, 2, 2.5, 5, 10) if m * mag >= raw] or [mag * 10])
    first = math.floor(lo / step) * step
    ticks, v = [], first
    while v <= hi + step * 0.5:
        ticks.append(round(v, 10))
        v += step
    return ticks

def _panel(title, bands, prior, curr, lo, hi, ticks, m, dark=False):
    W, H = 300, 188
    L, R, T, B = 46, 10, 26, 40
    pw, ph = W - L - R, H - T - B
    y = lambda v: T + ph - (v - lo) / (hi - lo) * ph
    out = ['<svg viewBox="0 0 %d %d" role="img" aria-label="%s">' % (W, H, escape(title, quote=True))]
    out.append('<text class="pt" x="0" y="12">%s</text>' % escape(title))
    for t in ticks:                                   # recessive grid
        yy = y(t)
        out.append('<line class="grid" x1="%d" y1="%.1f" x2="%d" y2="%.1f"/>' % (L, yy, W - R, yy))
        out.append('<text class="tick" x="%d" y="%.1f">%s</text>' % (L - 6, yy + 3, _tick_txt(t, m)))
    if lo < 0 < hi:
        out.append('<line class="zero" x1="%d" y1="%.1f" x2="%d" y2="%.1f"/>' % (L, y(0), W - R, y(0)))
    base = y(0) if lo < 0 < hi else y(lo)
    gw = pw / max(len(bands), 1)
    bw = min(20.0, (gw - 14) / 2)
    for gi, (band, idx) in enumerate(bands):
        cx = L + gw * gi + gw / 2
        for si, (kind, val) in enumerate((("curr", curr[gi]), ("prior", prior[gi]))):
            if _isna(val):
                continue
            bx = cx - bw - 1 + si * (bw + 2)          # 2px surface gap between bars
            top, bot = min(y(val), base), max(y(val), base)
            out.append('<rect class="bar %s" x="%.1f" y="%.1f" width="%.1f" height="%.1f" rx="4">'
                       '<title>%s %s - %s: %s</title></rect>'
                       % (kind, bx, top, bw, max(bot - top, 1.5),
                          escape(title), escape(band),
                          PRIOR_LABEL if kind == "prior" else CURR_LABEL,
                          _plain(val, m["py"])))
        out.append('<text class="band" x="%.1f" y="%d">%s</text>' % (cx, H - 20, escape(band)))
    out.append("</svg>")
    return "".join(out)

def _tick_txt(v, m):
    if m["py"].endswith("%}"):
        return ("%.0f%%" % (v * 100)) if abs(v * 100) >= 1 or v == 0 else ("%.1f%%" % (v * 100))
    if abs(v) >= 1000:
        return "%.0fk" % (v / 1000.0)
    return "%g" % round(v, 2)

def build_charts(df, isroll):
    fams = _families(df, isroll)
    if not fams:
        return ""
    blocks = []
    for mi, m in enumerate(METRICS):
        pri = list(df[(m["label"], "Prior")])
        cur = list(df[(m["label"], "Curr")])
        vals = [v for v in pri + cur if not _isna(v)]
        if not vals:
            continue
        lo, hi = min(vals), max(vals)
        lo = min(lo, 0.0) if lo < 0 else 0.0          # bars are read from a zero baseline
        hi = hi * 1.08 if hi > 0 else 0.0
        ticks = _nice_ticks(lo, hi)
        lo, hi = min(lo, ticks[0]), max(hi, ticks[-1])
        panels = [_panel(k, bands, [pri[i] for _, i in bands], [cur[i] for _, i in bands],
                         lo, hi, ticks, m)
                  for k, bands in fams]
        blocks.append('<div class="panels" data-metric="%d"%s>%s</div>'
                      % (mi, "" if mi == 0 else ' hidden', "".join(panels)))
    opts = "".join('<option value="%d"%s>%s</option>'
                   % (mi, " selected" if mi == 0 else "", escape(m["label"]))
                   for mi, m in enumerate(METRICS))
    return """
<section class="charts">
  <div class="chead">
    <div>
      <h2>Each product by vantage band</h2>
      <p>%CURRL% against %PRIORL% for every band inside a product. All panels share
      one scale, so panel heights are comparable across products.</p>
    </div>
    <div class="ctl">
      <label for="metric">Show</label>
      <select id="metric">%OPTS%</select>
      <span class="key"><i class="sw curr"></i>%CURRL%<i class="sw prior"></i>%PRIORL%</span>
    </div>
  </div>
  %BLOCKS%
</section>
""".replace("%OPTS%", opts).replace("%BLOCKS%", "".join(blocks)) \
   .replace("%CURRL%", escape(CURR_LABEL)).replace("%PRIORL%", escape(PRIOR_LABEL))

def build_page(df, sample=False, fragment=False, kinds=None):
    groups = _groups()
    n = len(df)
    kinds = ["segment"] * n if kinds is None else list(kinds)
    isroll = [k != "segment" for k in kinds]
    n_seg = kinds.count("segment")
    n_roll = kinds.count("rollup")
    n_tot = kinds.count("total")
    first_total = kinds.index("total") if n_tot else -1
    first_excl = kinds.index("excluded") if "excluded" in kinds else -1
    seg_only = [i for i in range(n) if not isroll[i]]
    tvals = {m["label"]: _tvals_split(list(df[(m["label"], "Var")]), m["good"], kinds)
             for m in METRICS}
    by_label = {m["label"]: m for m in METRICS}
    stamp = datetime.datetime.now().strftime("%d %B %Y, %H:%M")

    # ---- header, three tiers: family / metric / prior-curr-var -----------------
    h = []
    h.append('<div class="panel"><div class="scroll"><table>')
    h.append("<thead>")
    h.append('<tr><th class="fam corner stick sortable" rowspan="3" data-col="0" '
             'scope="col">Segment<span class="arrow">&#9650;</span></th>'
             '<th class="fam" rowspan="3" scope="col">Comments</th>')
    for name, mets in groups:
        h.append('<th class="fam fam-edge" colspan="%d" scope="colgroup">%s</th>'
                 % (3 * len(mets), escape(name)))
    h.append("</tr><tr>")
    for name, mets in groups:
        for k, m in enumerate(mets):
            edge = " fam-edge" if k == 0 else ""
            h.append('<th class="met%s" colspan="3" scope="colgroup">%s</th>'
                     % (edge, escape(m["label"])))
    h.append("</tr><tr>")
    col = 2
    for name, mets in groups:
        for k, m in enumerate(mets):
            for j, sub in enumerate(SUBCOLS):
                edge = " fam-edge" if (k == 0 and j == 0) else ""
                h.append('<th class="sub sortable%s" data-col="%d" scope="col">%s'
                         '<span class="arrow">&#9650;</span></th>'
                         % (edge, col, escape(SUBCOL_LABELS[sub])))
                col += 1
    h.append("</tr></thead><tbody>")

    # ---- body -----------------------------------------------------------------
    for r in range(n):
        seg = str(df[("Segment", "")].iloc[r])
        kind = kinds[r]
        chip = {"total": '<span class="chip">campaign total</span>',
                "excluded": '<span class="chip">excluded</span>',
                "rollup": '<span class="chip">roll-up</span>'}.get(kind, "")
        cls = {"total": "roll total" + (" first" if r == first_total else ""),
               "excluded": "gone" + (" first" if r == first_excl else ""),
               "rollup": "roll"}.get(kind, "")
        note = str(df[("Comments", "")].iloc[r] or "")
        h.append('<tr%s><td class="seg stick" data-v="%s">%s%s</td>'
                 '<td class="note">%s</td>'
                 % ((' class="%s"' % cls) if cls else "",
                    escape(seg, quote=True), escape(seg), chip, escape(note)))
        for name, mets in groups:
            for k, m in enumerate(mets):
                edge = " fam-edge" if k == 0 else ""
                p = df[(m["label"], "Prior")].iloc[r]
                c = df[(m["label"], "Curr")].iloc[r]
                v = df[(m["label"], "Var")].iloc[r]
                h.append('<td class="num cur%s" data-v="%s">%s</td>'
                         % (edge, "" if _isna(c) else c, _plain(c, m["py"])))
                h.append('<td class="num" data-v="%s">%s</td>'
                         % ("" if _isna(p) else p, _plain(p, m["py"])))
                t = tvals[m["label"]][r]
                if t is None:
                    h.append('<td class="var blank" data-v="">-</td>')
                else:
                    # no glyph: the signed number carries the direction on its own,
                    # so the reading never depends on telling green from red
                    word = "worse" if t > 0.5 else ("better" if t < 0.5 else "unchanged")
                    h.append('<td class="var" data-t="%.4f" data-v="%s" style="background:%s">'
                             '%s<span class="sr"> (%s)</span></td>'
                             % (t, v, _fill(t), _signed(v, m["py"]), word))
        h.append("</tr>")
    h.append("</tbody></table></div></div>")
    table = "".join(h)

    # ---- summary boxes -------------------------------------------------------
    # Volume is a count, so it sums. The other four are ratios or per-account
    # averages and cannot be added up, so each is a VOLUME-WEIGHTED average
    # across the segments - the same thing as dividing the portfolio total by the
    # portfolio's accounts. Roll-up rows are excluded so nothing is counted twice.
    def _seg(col):
        return [df[col].iloc[i] for i in seg_only]

    def _wavg(vals, wts):
        num = den = 0.0
        for v, w in zip(vals, wts):
            if _isna(v) or _isna(w):
                continue
            num += float(v) * float(w)
            den += float(w)
        return num / den if den else NAN_

    sp = globals().get("SUMMARY_PRIOR")
    sc = globals().get("SUMMARY_CURR")

    def _box(key, label, weighted):
        if key not in by_label:
            return
        m = by_label[key]
        if sc is not None and not _isna(sc.get(m["label"])):
            c = sc[m["label"]]
            p = sp.get(m["label"]) if sp else NAN_
            note = '<span class="w">from %s</span>' % escape(SUMMARY_SHEET)
        elif weighted and "Volume" in by_label:
            w = by_label["Volume"]["label"]
            p = _wavg(_seg((m["label"], "Prior")), _seg((w, "Prior")))
            c = _wavg(_seg((m["label"], "Curr")),  _seg((w, "Curr")))
            note = '<span class="w">volume-weighted</span>'
        else:
            p, c = _sum(_seg((m["label"], "Prior"))), _sum(_seg((m["label"], "Curr")))
            note = '<span class="w">sum of %d segments</span>' % n_seg
        if _isna(c):
            return
        d = c - p
        good = (d > 0) if m["good"] == "up" else (d < 0)
        cls = "up" if (d and good) else ("down" if d else "")
        tiles.append('<div class="stat"><span class="k">%s</span>'
                     '<span class="v">%s</span>'
                     '<span class="d %s">%s vs %s</span>%s</div>'
                     % (escape(label), _plain(c, m["py"]), cls, _signed(d, m["py"]),
                        escape(PRIOR_LABEL), note))

    tiles = []
    _box("Volume",          "Total volume",     False)
    _box("Yr1 Unit Loss",   "Yr1 unit loss",    True)
    _box("Yr3 ROA",         "Yr3 ROA",          True)
    _box("CPA",             "CPA",              True)
    _box("Avg Credit Line", "Avg credit line",  True)

    # ---- page ----------------------------------------------------------------
    flag = ('<div class="flag"><b>Sample data.</b> These figures come from placeholder '
            'workbooks, not the real model - regenerate before sharing this page.</div>'
            if sample else "")

    page = """
%FLAG%
<header class="mast">
  <span class="eyebrow">%CURRL% against %PRIORL% &middot; %STAMP%</span>
  <h1>Segment Heat Map</h1>
</header>

<section class="stats">%TILES%</section>

<div class="bar">
  <div class="ctl"><span>Variance colours</span>
    <span class="seg" role="group" aria-label="Colour ramp">
      <button type="button" data-ramp="excel" aria-pressed="true">Workbook</button>
      <button type="button" data-ramp="safe" aria-pressed="false">Colour-blind safe</button>
    </span>
  </div>
  <div class="ctl"><button type="button" id="reset" class="seg" style="padding:5px 11px;cursor:pointer">Reset order</button></div>
</div>

%TABLE%

%CHARTS%

<div class="sig">
  <span>%CURRL%: <code>%CURR%</code></span>
  <span>%PRIORL%: <code>%PRIOR%</code></span>
  <span>Generated %STAMP%</span>
</div>
"""
    page = (page.replace("%FLAG%", flag).replace("%STAMP%", stamp)
                .replace("%TILES%", "".join(tiles))
                .replace("%TABLE%", table)
                .replace("%CHARTS%", build_charts(df, isroll))
                .replace("%PRIOR%", escape(os.path.basename(PRIOR_WORKBOOK)))
                .replace("%CURR%", escape(os.path.basename(CURRENT_WORKBOOK)))
                .replace("%CURRL%", escape(CURR_LABEL))
                .replace("%PRIORL%", escape(PRIOR_LABEL)))

    js = _JS.replace("%RAMPS%", json.dumps(RAMPS))
    head = ('<title>Segment Heat Map</title>\n'
            '<link rel="stylesheet" href="https://fonts.googleapis.com/css2?'
            'family=Newsreader:opsz,wght@6..72,400;6..72,600&'
            'family=IBM+Plex+Sans:wght@400;500;600&'
            'family=IBM+Plex+Mono:wght@400;500&display=swap">\n'
            '<style>%s</style>' % _CSS)
    body = '<main class="wrap">%s</main>\n<script>%s</script>' % (page, js)

    if fragment:          # what the Artifact publisher wants: no html/head/body
        return head + "\n" + body
    return ('<!doctype html>\n<html lang="en">\n<head>\n<meta charset="utf-8">\n'
            '<meta name="viewport" content="width=device-width,initial-scale=1">\n'
            '%s\n</head>\n<body>\n%s\n</body>\n</html>\n' % (head, body))

def write_html(df, path="segment_heatmap.html", sample=False, kinds=None):
    full = build_page(df, sample=sample, fragment=False, kinds=kinds)
    frag = build_page(df, sample=sample, fragment=True, kinds=kinds)
    alt = os.path.splitext(path)[0] + "_artifact.html"
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(full)
    with open(alt, "w", encoding="utf-8") as fh:
        fh.write(frag)
    return os.path.abspath(path), os.path.abspath(alt)

page_path, frag_path = write_html(heat, "segment_heatmap.html", sample=False,
                                  kinds=ROW_KIND)
print("open this one:      ", page_path)
print("publish this one:   ", frag_path)